# Ariel data challenge submission

## 1. Notebook set up

In [2]:
# Standard library imports
import os
import time

# Third party imports
import numpy as np
import pandas as pd
import tensorflow as tf

# Project imports
from ariel_data_preprocessing.data_preprocessing import DataProcessor

mode = 'submission'

if mode == 'testing':
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    INPUT_DIRECTORY = 'data/raw'
    OUTPUT_DIRECTORY = 'data/processed'
    REBUILD_DATA = True
    N_PLANETS = 10
    MODEL = 'data/models/ariel-cnn-8.1M-2ksteps-tf2.11.keras'

elif mode == 'submission':
    INPUT_DIRECTORY = '/kaggle/input/ariel-data-challenge-2025'
    OUTPUT_DIRECTORY = '/kaggle/working'
    REBUILD_DATA = True
    N_PLANETS = -1
    MODEL = '/kaggle/input/ariel-cnn/ariel-cnn-8.1M-862ksteps.keras'

2025-09-22 22:51:27.849303: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758581488.114412     102 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758581488.192169     102 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## 2. Data preparation

### 2.1. Preprocess the raw data

In [3]:
data_processor = DataProcessor(
    input_data_path=INPUT_DIRECTORY,
    output_data_path=OUTPUT_DIRECTORY,
    output_filename='test.h5',
    n_cpus=4,
    downsample_fgs=True,
    n_planets=N_PLANETS,
    mode='test'
)

In [4]:
if REBUILD_DATA:
    start_time = time.time()
    data_processor.run()
    end_time = time.time()

    print(f'Data preprocessing completed in {(end_time - start_time)/60:.2f} minutes')

Data preprocessing completed in 0.39 minutes


In [5]:
print(f'Total test planets: {len(data_processor.planet_list)}')

Total test planets: 1


### 2.2. Initialize data generator

In [6]:
data_processor.initialize_data_generators(
    sample_size=372,
    n_samples=10
)

2025-09-22 22:52:31.766832: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


### 2.3. Create dataset

In [7]:
testing_data = data_processor.testing.take(len(data_processor.planet_list))
signals = np.array([element.numpy() for element in testing_data])

print(f'Signals shape: {signals.shape}')

Signals shape: (1, 10, 372, 283)


## 3. Predictions

In [8]:
model = tf.keras.models.load_model(MODEL)

spectrum_predictions = []

for planet in signals:
    spectrum_predictions.append(model.predict(planet, batch_size=10, verbose=0))

spectrum_predictions = np.array(spectrum_predictions)
spectrum_predictions_avg = np.mean(spectrum_predictions, axis=1)
spectrum_predictions_std = np.std(spectrum_predictions, axis=1)

print(f'Spectrum predictions shape: {spectrum_predictions.shape}')
print(f'Spectrum predictions avg shape: {spectrum_predictions_avg.shape}')
print(f'Spectrum predictions std shape: {spectrum_predictions_std.shape}')

I0000 00:00:1758581565.160746     201 service.cc:148] XLA service 0x7ee220007c80 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1758581565.161795     201 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1758581565.658974     201 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Spectrum predictions shape: (1, 10, 283)
Spectrum predictions avg shape: (1, 283)
Spectrum predictions std shape: (1, 283)


## 4. Submission file

In [10]:
submission = np.concatenate(
    (spectrum_predictions_avg, spectrum_predictions_std),
    axis=1
)

submission_df = pd.DataFrame(submission)
submission_df.index = data_processor.planet_list
submission_df.to_csv('submission.csv', header=False)

print(f'Submission shape: {submission.shape}')

Submission shape: (1, 566)
